# Gradient Reversal Layer (GRL)

The **Gradient Reversal Layer (GRL)** is used in **Domain-Adversarial Neural Networks (DANN)** for **domain adaptation**.

## Main Idea

GRL behaves differently during the **forward** and **backward** passes.

- **Forward pass:** It does nothing; the input is passed unchanged.
- **Backward pass:** It reverses and scales the gradient by `-λ`.

### Mathematically

**Forward:**

$$
GRL(x) = x
$$

**Backward:**

$$
\frac{\partial L}{\partial x}
=
-\lambda
\frac{\partial L}{\partial GRL(x)}
$$

---

## Why Do We Need GRL?

In **domain adaptation**, the **feature extractor** should learn features that:

1. Are useful for the **main/task classifier**.
2. Do not contain information that makes it easy to distinguish between the **source domain** and the **target domain**.

### DANN Architecture

```text
                    ┌──────────────→ Task Classifier → Task Loss
                    │
Input → Feature Extractor → Features
                    │
                    └→ GRL → Domain Classifier → Domain Loss

In [ ]:
import torch
from torch.autograd import Function
#Function is a PyTorch class for creating custom autograd operations


class GradientReversalFunction(Function): #"I am defining my own autograd operation."

    @staticmethod  # What happens during forward?
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_

        # Forward: do nothing
        return x

    @staticmethod  # What happens during backpropagation?
    def backward(ctx, grad_output):
        # Backward: reverse and scale the gradient
        grad_input = -ctx.lambda_ * grad_output

        return grad_input, None


#We can create a small GRL module
class GradientReversal(torch.nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)


grl = GradientReversal(lambda_=0.5)


In [ ]:

class Model(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.feature_extractor = ...
        self.label_predictor = ...
        self.domain_classifier = ...

        self.grl = GradientReversal(lambda_=0.5)

    def forward(self, x):

        # Extract features
        f = self.feature_extractor(x)

        # Label prediction
        y_pred = self.label_predictor(f)

        # Domain prediction through GRL
        d_pred = self.domain_classifier(self.grl(f))

        return y_pred, d_pred